# Reproducing Kalugin et al. (2026) from dandiset 001074

Paper. Kalugin PN, et al. (2026). Simultaneous, real-time tracking of many neuromodulatory
signals with multiplexed optical recording of sensors on a micro-endoscope. *Neuron*.
[doi:10.1016/j.neuron.2026.07.008](https://doi.org/10.1016/j.neuron.2026.07.008). The claims file
was written from the accepted author manuscript in PubMed Central (PMC13568791) with its
supplement (Figures S1 to S6); the version of record was not read.

Dandiset. [DANDI:001074](https://dandiarchive.org/dandiset/001074/0.260928.0807), version
0.260928.0807. The paper cites the dandiset without a version. Three versions are published
(0.260804.2042, 0.260807.0137, 0.260928.0807); their asset lists hold the same 713 paths with the
same blobs and sizes (checked through the API while preparing), so they do not differ in anything
the analysis uses. The most recent is pinned, as the protocol asks when the paper cites none.

What the dandiset holds. For each MORSE recording, the CaImAn output of the paper's pipeline:
a trace per cell mask (the denoised trace and its residual) and the 3D masks. There are no sensor
labels, no well, wash-on, injection or stimulation times, no region or experiment labels, and no
deformation fields, raw imaging, photometry, ELISA, histology or choroid plexus traces. Subjects
are named by date and probe, with the context (`wells`, `slice`, `mouse`).

Outside input. The well order of the ten revision probes' identification and dose-response
dipping, the mixture concentrations, and the baseline and normalization wells of each fit come
from the paper's code, github.com/pkalugin/kalugin-et-al-morse at commit
fa4526ac02eaa2c5a24c4cfdcaab5c36ed5ae4ed (`6_trace_analysis/morse_figs3_rev.ipynb`). They name
the frame blocks of recordings the dandiset holds and bring in no measurements, so the claims
that use them are `with_labels` and depend on the link claim `link-revision-well-order`.

| Claim | Tested here |
|---|---|
| link-revision-well-order (link, added by the author) | tested |
| fig1c-cells-per-gel | tested (curated masks only) |
| fig1g-sensor-yield, fig1g-all-ten-sensors-respond, fig1h-crosstalk-pairs | tested |
| table1-ec50-on-target, table1-ec50-off-target, table1-peak-dff-morse | tested |
| fig1j-ec50-consistent-across-probes, figs3e-tau-on-under-one-minute, figs3f-depth-delay | tested |
| figs2f-cells-tracked | tested |
| fig1d, fig1f, methods-revision-probe-stability, figs2c, slices-probe-stability | blocked, data missing (deformation fields, raw imaging) |
| figs6ef, fig4-ex-vivo-elevations, fig4-ex-vivo-two-orders-lower, figs6j, fig5gh, fig6, fig6c, figs6a | blocked, data missing (photometry, ELISA and mass spectrometry, choroid plexus, histology) |
| the 16 slice claims (fig2c to fig2l, figs4de, figs5c to figs5i, slices-no-peptide-release) | blocked, metadata insufficient |
| fig3df, fig3g, fig3h, fig4-ip-csf-concentrations, fig5ab, fig5cd | blocked, metadata insufficient |

In [ ]:
import time
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import morse as M

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()
print("dandiset", M.DANDISET, "version", M.VERSION)

## Data check

The survey reads the shapes and metadata of all 713 files, and one value of each trace table to
find files deposited twice.

In [ ]:
t0 = time.time()
S = pd.DataFrame(M.survey())
S["kind"] = S.path.map(M.kind_of)
S["context"] = S.subject.str.extract(r"(wells|slice|mouse)")[0].fillna("none")
print(f"{len(S)} files read in {time.time() - t0:.0f} s; {S.subject.nunique()} subjects")
print("\nFiles by context and kind:")
print(pd.crosstab(S.kind, S.context).to_string())
print("\nMetadata values over all files:")
for c in ["rate_hz", "dt", "t0", "indicator", "start", "accepted", "rejected", "acquisition", "stimulus"]:
    print(f"  {c}: {sorted(S[c].unique().tolist())[:6]}")
print("\nFrames per file: min", S.frames.min(), "max", S.frames.max(),
      "; cells per file: min", S.cells.min(), "max", S.cells.max())
print("Mask volume planes (z) per file:", sorted(Counter(s[2] for s in S.mask_shape).items()))

Every file states an imaging rate of 30 Hz, timestamps from 0 in steps of 1/30 s, the indicator
"OGB-1", and the same session start (2023-01-01 12:30). These are placeholders: the Methods give
about 0.5 volumes per second, the paper's code uses 31 volumes per minute (`mean_factor` 31), and
the sensors are GRAB sensors. The frame index is the only clock the files have; at 31 frames per
minute the recordings run from 1.2 to 314 minutes. No mask is marked accepted or rejected.
No file has acquisition or stimulus data. The mask volumes have 15 to 26 planes, where the Methods
describe 30 planes per volume.

In [ ]:
S["frames_mod_93"] = S.frames % M.FRAMES_PER_WELL
w = S[S.context == "wells"]
print("Dipping (wells) files whose frame count is a multiple of 93 (one well = 3 min):",
      int((w.frames_mod_93 == 0).sum()), "of", len(w))
print("Frame counts of the dipping files that are not:", sorted(w.frames[w.frames_mod_93 != 0].unique().tolist()))
dup = S.groupby(["frames", "cells", "first_values"]).path.apply(list)
dup = dup[dup.str.len() > 1]
print(f"\nGroups of files with identical shape and first trace values: {len(dup)}")
for v in dup:
    print("  ", [p.split("/")[-1].replace("_ophys.nwb", "").replace("ses-ch0-means-movie-nobg-cnmf-", "") for p in v])
print("\nThe 67 files of subject '0' (no name):", int((S.subject == "0").sum()),
      "; of them, duplicates of a named file:",
      sum(any(p.startswith("sub-0/") for p in v) and any(not p.startswith("sub-0/") for p in v) for v in dup))

All dipping files but a few have frame counts that are multiples of 93, consistent with 3-minute
wells at 31 frames per minute. Files deposited twice are listed above: the unnamed subject `0`
repeats named files, and the slice and wells001 files of probe 250918-SL1ZS5 are identical, so one
of the two recordings of that probe is missing and the other is deposited under both names. Each
recording is counted once below. The masks are stored scrambled; the cell below shows it on one
mask.

In [ ]:
p = M.file_path("251003-PL65", "wells001")
with M.open_nwb(p) as f:
    m = f["processing/ophys/ImageSegmentation/PlaneSegmentation/image_mask"][0]
nz = np.argwhere(m > 0)
print("mask 0 of", p.split("/")[-1], "shape", m.shape)
print("  as stored: nonzero voxels", len(nz), "bounding box", nz.min(0).tolist(), "to", nz.max(0).tolist())
v = m.ravel().reshape(m.shape, order="F")
nz = np.argwhere(v > 0)
print("  reordered (Fortran order): bounding box", nz.min(0).tolist(), "to", nz.max(0).tolist())
print("  per-plane weight as stored:", np.round(m.sum(axis=(0, 1)), 2).tolist())
print("  per-plane weight reordered:", np.round(v.sum(axis=(0, 1)), 2).tolist())

Read as stored, the mask covers the full width of the volume with the same weight in every plane.
Reordered as a Fortran-order volume it is a compact blob of about 25 by 30 pixels over about 12
planes, as a cell should be. The masks were written from CaImAn's column-major vectors without
the reordering; every depth below uses the reordered masks.

## The revision probes and the well order (link-revision-well-order)

> The code's well lists for the ten revision probes (identification: AVP, 5-HT, NE, ACh, ATP, OXT,
> HA, DA, CCK, MT, PBS; dose response: the A5NAA mixture at seven 10-fold steps, the OHDCM mixture
> at seven steps, PBS) apply to the dandiset's wells001 and wells101 recordings of those probes.

The matched files of the identification (wells001) and dose (wells101) recordings are used, as in
morse_figs3_rev cells 5 and 7. Traces are C plus the residual (the code's `CY_matched_raw`).

In [ ]:
t0 = time.time()
R = M.load_rev()
rows = []
for p in M.REV_PROBES:
    rows.append({"probe": p, "id_cells": R[p]["id"].shape[0], "id_frames": R[p]["id"].shape[1],
                 "id_wells": R[p]["id"].shape[1] / 93, "dose_cells": R[p]["dose"].shape[0],
                 "dose_frames": R[p]["dose"].shape[1], "dose_wells": R[p]["dose"].shape[1] / 93})
rev = pd.DataFrame(rows)
print(f"loaded in {time.time() - t0:.0f} s")
print(rev.to_string(index=False))
print("cells over the ten probes:", rev.id_cells.sum())

In [ ]:
pre = [M.id_preprocess(R[p]["id"]) for p in M.REV_PROBES]
PN = np.vstack([x[1] for x in pre])
act = M.derivative(PN).argmax(axis=1)
within = act % 93
print("cells:", len(act), "; exponential background fitted for", int(sum(x[2].sum() for x in pre)),
      "(the rest activate within the first two wells)")
print("Frame of each cell's steepest rise within its well (0 = onset), counts by bin:")
print(pd.Series(pd.cut(within, [-1, 10, 20, 30, 46, 62, 92])).value_counts(sort=False).to_string())
frac = float(np.mean(within < 31))
print(f"Fraction rising in the first minute of a well: {frac:.2f} (one third if rises fell anywhere in a well)")
for shift in [0, 15, 31, 46]:
    fr = float(np.mean(((act - shift) % 93) < 31))
    print(f"  with well onsets shifted by {shift:2d} frames: {fr:.2f}")

Sensor labels (SENSOR IDENTITY of the claims file, the dataset giving none). The Methods'
procedure: dF/F0 (Eq. 27), the jump artefact removed, an exponential background fitted before
activation, peak normalization (Eq. 28), a Savitzky-Golay derivative over 1 minute, HDBSCAN
(min_cluster_size 3) on correlation distance, and clusters matched to ligands by correlation
with the time course of ligand administration. Reading (a) pools the cells of all ten probes in
one clustering; reading (b) adds, for cells (a) leaves unlabelled, the label of their cluster in a
clustering of their own probe, as the code's second pass does (morse_figs3_rev cells 62-69). The
code then corrects clusters by hand (cells 73-78), which cannot be repeated.

In [ ]:
la, lb, grp, der = M.identify([x[1] for x in pre])
LAB = lb
print("labelled cells, reading (a):", int((la != "").sum()), " reading (b):", int((lb != "").sum()), "of", len(lb))
tab = pd.DataFrame({"a": pd.Series(la[la != ""]).value_counts(), "b": pd.Series(lb[lb != ""]).value_counts()}).reindex(M.SENSORS)
print(tab.to_string())

In [ ]:
DX = np.vstack([M.remove_jumps(M.dff(R[p]["dose"])) for p in M.REV_PROBES])
PLm = M.well_plateaus(DX)
rows = []
for s in M.SENSORS:
    r = LAB == s
    a = PLm[r][:, 0:7].max(axis=1) - PLm[r][:, 0]
    o = PLm[r][:, 7:14].max(axis=1) - PLm[r][:, 7]
    own = "A" if s in M.MIX_A else "O"
    hit = (a > o) if own == "A" else (o > a)
    rows.append({"sensor": s, "mixture": own, "cells": int(r.sum()), "larger rise in own mixture": int(hit.sum()),
                 "fraction": round(float(hit.mean()), 2) if r.sum() else np.nan})
link = pd.DataFrame(rows)
print("Dose recording: rise of each cell's well plateaus (mean of the third minute) over the first well of each mixture block")
print(link.to_string(index=False))
nx = link[~link.sensor.isin(["AVP", "OXT", "NE", "DA"])]
print("Fraction over sensors without crosstalk:", round(nx["larger rise in own mixture"].sum() / nx.cells.sum(), 2))
print("Fraction over all labelled cells:", round(link["larger rise in own mixture"].sum() / link.cells.sum(), 2))

**Result: reproduced.** All ten identification recordings have 1023 frames (11 wells of 93) and
all ten dose recordings 1395 frames (15 wells), as the code's well lists require. 86% of cells rise
most steeply in the first minute of a well, against 23%, 11% and 3% with the well onsets shifted by
15, 31 and 46 frames. In the dose recording, 79% of the labelled cells (77% for the sensors without
crosstalk, and 73% to 85% for each of them) rise more in the mixture block that holds their
ligand; the DA sensor cells, which also respond to the NE of the other block, do so in 41%.

## fig1c-cells-per-gel

> Imaging a MORSE hydrogel through the GRIN lens shows 100 to 200 green sensor cells spread
> through the full thickness of the gel.

Reading (a) of the criterion, the CNMF components before curation, is not in the dandiset: its
files hold the curated masks (`mr-raw`, merged and refit), the masks after a further manual merge
(`mr-m-raw`) and the matched masks. Reading (b) is tested on the `mr-raw` files, one per recording,
and shown also on `mr-m-raw`. Depth uses the reordered masks of the `mr-raw` identification
recordings of the revision probes that have one; the gel thickness is read as the imaged z range,
an assumption (the claims file does not say how the thickness is bounded).

In [ ]:
U = S[(S.subject != "0")].copy()
U = U[~((U.subject == "250918-SL1ZS5-wells001"))]  # deposited identical to its slice recording
U["probe"] = U.subject.str.replace(r"-(wells\d*|slice|mouse\d?)$", "", regex=True)
for k in ["mr-raw", "mr-m-raw"]:
    d = U[U.kind == k]
    per_probe = d.groupby("probe").cells.mean()
    inr = ((per_probe >= 90) & (per_probe <= 220))
    print(f"{k}: {len(d)} recordings of {d.probe.nunique()} probes; masks per recording median {d.cells.median():.0f}, "
          f"range {d.cells.min()} to {d.cells.max()}")
    print(f"   probes whose mean count lies in 90-220: {int(inr.sum())} of {len(inr)} ({inr.mean():.2f})")
    print("   by context:", d.groupby("context").cells.agg(["count", "median", "min", "max"]).to_dict("index"))

In [ ]:
t0 = time.time()
rows = []
for p in M.REV_PROBES:
    path = M.file_path(p, "wells001", "mr-raw")
    if path not in M.assets():
        continue
    wz = M.mask_z_profiles(path)
    z = M.z_centre(wz)
    nz = wz.shape[1]
    third = np.floor(z / (nz / 3)).clip(0, 2).astype(int)
    c = np.bincount(third, minlength=3)
    rows.append({"recording": p + " wells001", "masks": len(z), "planes": nz, "z min": round(z.min(), 1),
                 "z max": round(z.max(), 1), "top third": c[0], "middle third": c[1], "bottom third": c[2]})
depth1c = pd.DataFrame(rows)
print(f"masks read in {time.time() - t0:.0f} s")
print(depth1c.to_string(index=False))

**Result: reproduced, under reading (b) and the assumption that the imaged z range is the gel
thickness.** Over 59 recordings of 43 probes the curated masks number 21 to 205 per recording
(median 128), and 38 of 43 probes (0.88) have a mean count within 90 to 220; with the further
merged masks (`mr-m-raw`), 31 of 38. In each of the five identification recordings read, mask
centres fall in the top (5 to 21), middle (44 to 88) and bottom (27 to 67) thirds of the imaged
volume. Reading (a), the CNMF components before curation, cannot be tested: they are not in the
dandiset.

## fig1g-sensor-yield

> Ten ten-sensor probes yielded 506 identified single-cell masks; per probe a median of 9 sensors
> were active (IQR 8 to 9), with on average 6.02 cells per sensor per probe.

A sensor is active on a probe when at least one cell carries its label; cells per sensor per probe
is the number of labelled cells over the number of (probe, sensor) pairs with a cell
(morse_figs3_rev cells 139-144). The code prints sensors per probe 5, 8, 9, 10, 6, 9, 10, 9, 9, 9
(cell 140), shown for comparison. The derivative window of the code (30 frames) is shown as a
variant of the Methods' 1 minute (31 frames).

In [ ]:
code_counts = [5, 8, 9, 10, 6, 9, 10, 9, 9, 9]
rows = []
for name, lab in [("(a) pooled", la), ("(b) pooled + per probe", lb)]:
    y = M.yield_stats(lab, grp)
    rows.append({"reading": name, **{k: (round(v, 2) if isinstance(v, float) else v) for k, v in y.items() if k != "sensors_per_probe"},
                 "sensors per probe": y["sensors_per_probe"]})
la30, lb30, _, _ = M.identify([x[1] for x in pre], window=30)
for name, lab in [("(a) window 30", la30), ("(b) window 30", lb30)]:
    y = M.yield_stats(lab, grp)
    rows.append({"reading": name, **{k: (round(v, 2) if isinstance(v, float) else v) for k, v in y.items() if k != "sensors_per_probe"},
                 "sensors per probe": y["sensors_per_probe"]})
print(pd.DataFrame(rows).to_string(index=False))
print("code's sensors per probe:", code_counts)
print("Expected: 506 cells (455-557), median 9 (8.1-9.9), 25th percentile 8 (7.2-8.8), 75th 9 (8.1-9.9), 6.02 (5.42-6.62)")
print("\nCells per sensor and probe, reading (b):")
print(pd.crosstab(pd.Series(lb[lb != ""], name="sensor"), pd.Series(np.array(M.REV_PROBES)[grp[lb != ""]], name="probe")).reindex(M.SENSORS).fillna(0).astype(int).to_string())

**Result: partially reproduced, resting on the labelling.** The sensors per probe match: median 9
(reading b; 8.5 in reading a), 25th percentile 8 and 75th 9, and the per-probe counts
[5, 8, 9, 10, 5, 9, 10, 9, 8, 9] differ from the code's [5, 8, 9, 10, 6, 9, 10, 9, 9, 9] by one
sensor on two probes. The cell counts do not: 559 labelled cells in reading (b), just above the 455
to 557 range, and 382 in reading (a); 6.82 and 4.72 cells per sensor per probe against 5.42 to
6.62. The code's 30-frame window changes these by at most 8 cells. The code's hand corrections,
which drop and move cells, are not repeated.

## fig1g-all-ten-sensors-respond

> Dipping ten-sensor probes into each ligand in turn gave rapid, strong and consistent responses
> in a subset of cells for each ligand, and all ten sensors were activated.

For each sensor, the mean over its cells of the peak-normalized trace; in which well is it highest
(the maximum within each well, and as a variant the mean over each well)? The criterion allows the
DA/NE and AVP/OXT partners. Because the labels are made from the time of each cell's steepest rise,
this test is not independent of the labelling; it shows where the mean signal peaks.

In [ ]:
partner = {"AVP": "OXT", "OXT": "AVP", "NE": "DA", "DA": "NE"}
rows = []
for s in M.SENSORS:
    r = LAB == s
    m = PN[r].mean(axis=0)
    wmax = [m[w * 93:(w + 1) * 93].max() for w in range(11)]
    wmean = [m[w * 93:(w + 1) * 93].mean() for w in range(11)]
    own = M.ID_WELLS.index(s)
    ok = {s, partner.get(s, s)}
    rows.append({"sensor": s, "cells": int(r.sum()), "probes": len(set(grp[r])),
                 "own-well max": round(wmax[own], 2), "highest well (max)": M.ID_WELLS[int(np.argmax(wmax))],
                 "its value": round(max(wmax), 2), "highest well (mean)": M.ID_WELLS[int(np.argmax(wmean))],
                 "own or partner (max)": M.ID_WELLS[int(np.argmax(wmax))] in ok,
                 "own, partner or next well (max)": M.ID_WELLS[int(np.argmax(wmax))] in ok | {M.ID_WELLS[own + 1]}})
resp = pd.DataFrame(rows)
print(resp.to_string(index=False))
print("sensors with cells on at least one probe:", int((resp.cells > 0).sum()), "of 10")
print("sensors whose mean peaks in their own (or partner's) well:", int(resp["own or partner (max)"].sum()), "of 10")
print("allowing the well after the ligand's (the code's choice for AVP, NE and CCK, cell 248):",
      int(resp["own, partner or next well (max)"].sum()), "of 10")

**Result: partially reproduced.** All ten sensors carry labels, each on 5 to 10 of the ten
probes. The mean peak-normalized trace of a sensor's cells is highest in its own or its partner's
well for 2 of 10 sensors (OXT, ATP); allowing the well after the ligand's, as the code does for AVP,
NE and CCK, for 7 of 10 (adding AVP, 5-HT, ACh, MT and HA). For CCK, NE and DA it is highest later
still. The second part of the criterion fails as written.

## fig1h-crosstalk-pairs

> Among the ten sensors, only the DA and NE sensors, and to a lesser extent the AVP and OXT
> sensors, show crosstalk in their ligand responses.

Per cell, the derivative of the peak-normalized trace (Savitzky-Golay, 1 minute, order 1; the
code's 10 frames as variant), divided by its maximum; the mean per sensor; the Pearson correlation
at zero lag between the ten means. The band-pass artefact variant of cell 219 is not run.

In [ ]:
for win in [31, 10]:
    d = np.vstack([M.derivative(x[1], win) for x in pre])
    d = d / d.max(axis=1, keepdims=True)
    means = np.array([d[LAB == s].mean(axis=0) for s in M.SENSORS])
    C = pd.DataFrame(np.corrcoef(means), index=M.SENSORS, columns=M.SENSORS)
    iu = np.triu_indices(10, 1)
    pairs = sorted([(C.values[i, j], M.SENSORS[i], M.SENSORS[j]) for i, j in zip(*iu)], reverse=True)
    print(f"window {win} frames: largest off-diagonal correlations")
    for v, a, b in pairs[:5]:
        print(f"   {a}-{b}: {v:.3f}")
    rank = [k for k, (v, a, b) in enumerate(pairs) if {a, b} == {"AVP", "OXT"}][0] + 1
    print(f"   AVP-OXT: {C.loc['AVP', 'OXT']:.3f}, rank {rank} of 45")
    if win == 31:
        print(C.round(2).to_string())

**Result: partially reproduced.** The DA-NE correlation of the sensors' mean derivatives is the
largest off-diagonal value (0.77 with the 1-minute window, 0.83 with the code's 10 frames). The
AVP-OXT correlation is not the second: it is 0.16, rank 12 of 45 (0.43, rank 14, with 10 frames),
below 5-HT-DA (0.46), DA-ACh (0.45), 5-HT-NE (0.37) and others.

## table1-ec50-on-target and table1-ec50-off-target

> The MORSE EC50 of each sensor for its own ligand, from consensus dose-response fits, is AVP
> 355 nM, OXT 2.19 uM, 5-HT 687 nM, CCK 245 nM, NE 4.04 uM, DA 1.70 uM, ACh 7.03 uM, ATP 75.2 uM,
> MT 22.0 nM and HA 6.39 uM.

> The crosstalk sensors respond to the partner ligand with EC50s of 28.7 uM (AVP sensor to OXT),
> 247 nM (OXT sensor to AVP), 9.35 uM (NE sensor to DA) and 8.20 uM (DA sensor to NE).

Dose recording of each labelled cell: dF/F0, jump artefact removed. Plateau per well: the mean of
the third minute (Methods), with the code's median of sorted frames 47-77 as variant. The
normalization is open in the Methods ("corrected with an exponentially decaying background fit and
then peak-normalized"); three readings are run: (i) the code's, the baseline-well plateau
subtracted and divided by the peak-well plateau, from the baseline well on (morse_figs3_rev cells
338-339, 365); (ii) the same with the code's plateau; (iii) Eq. 28 over the dose recording after
an exponential background fitted before activation, all seven wells. Eq. 9 with Hill coefficient
1 is fitted by least squares to the cells of all probes pooled. The code's noise cut and its
pooling with an older probe set for ACh, ATP, MT and HA: the noise cut is run as variant (iv)
(reading i on the cells it keeps, cells 343 and 365); the older probe set is not part of this
claim's ten probes and is not run. The code's per-cell background fits with hand-set windows
(cells 285-293) cannot be repeated.

In [ ]:
EXP_EC50 = {("AVP", "AVP"): 355, ("OXT", "OXT"): 2190, ("5HT", "5HT"): 687, ("CCK", "CCK"): 245, ("NE", "NE"): 4040,
            ("DA", "DA"): 1700, ("ACh", "ACh"): 7030, ("ATP", "ATP"): 75200, ("MT", "MT"): 22.0, ("HA", "HA"): 6390,
            ("AVP", "OXT"): 28700, ("OXT", "AVP"): 247, ("NE", "DA"): 9350, ("DA", "NE"): 8200}
PLc = M.well_plateaus(DX, "sorted_median")
rows = []
FITS = {}
for g in M.GROUP_ORDER:
    r = LAB == g[0]
    out = {"sensor": g[0], "ligand": g[1], "cells": int(r.sum()), "paper nM": EXP_EC50[g]}
    for name, PL in [("i", PLm), ("ii", PLc)]:
        c, y = M.group_points(PL[r], g, "code")
        kd, f, n = M.fit_group(c, y)
        FITS[(name, g)] = (kd, f)
        out[f"({name}) nM"] = round(kd * 1e9, 1)
        out[f"({name}) F"] = round(f, 2)
        out[f"({name}) within 10%"] = abs(kd * 1e9 - EXP_EC50[g]) <= 0.1 * EXP_EC50[g]
    keep = M.noise_keep(M.normalized_dose_trace(DX[r], g))
    c, y = M.group_points(PLm[r][keep], g, "code")
    kd, f, n = M.fit_group(c, y)
    FITS[("iv", g)] = (kd, f)
    out["(iv) cells"], out["(iv) nM"], out["(iv) F"] = int(keep.sum()), round(kd * 1e9, 1), round(f, 2)
    out["(iv) within 10%"] = abs(kd * 1e9 - EXP_EC50[g]) <= 0.1 * EXP_EC50[g]
    x = DX[r]
    b = M.block_of(g[1]) * 93
    a = b + M.derivative(M.peak_norm(x))[:, b:b + 651].argmax(axis=1)
    xb, _ = M.exp_background(x, a)
    c, y = M.group_points(M.well_plateaus(M.peak_norm(xb)), g, "eq28")
    kd, f, n = M.fit_group(c, y)
    out["(iii) nM"], out["(iii) F"] = round(kd * 1e9, 1), round(f, 2)
    out["(iii) within 10%"] = abs(kd * 1e9 - EXP_EC50[g]) <= 0.1 * EXP_EC50[g]
    rows.append(out)
EC = pd.DataFrame(rows)
print(EC.to_string(index=False))
on = EC[EC.sensor == EC.ligand]
offt = EC[EC.sensor != EC.ligand]
for name in ["i", "ii", "iii", "iv"]:
    print(f"reading ({name}): on-target within 10%: {int(on[f'({name}) within 10%'].sum())} of 10; "
          f"off-target: {int(offt[f'({name}) within 10%'].sum())} of 4; "
          f"median |log10 ratio| on-target {np.median(np.abs(np.log10(on[f'({name}) nM'] / on['paper nM']))):.2f}")

In [ ]:
print("Median normalized plateau per well (reading i), by group; concentrations in log10 M")
for g in M.GROUP_ORDER:
    r = LAB == g[0]
    c, y = M.group_points(PLm[r], g, "code")
    print(f"  {g[0]:>3} by {g[1]:<3} log10 c {np.round(np.log10(c), 0).astype(int).tolist()}  median y {np.round(np.nanmedian(y, axis=0), 2).tolist()}  "
          f"range of y {np.nanmin(y):.1f} to {np.nanmax(y):.1f}")

**Result: not reproduced.** With the code's normalization and the Methods' plateau (reading i),
2 of 10 EC50s are within 10% of Table 1 (OXT 2217 nM, CCK 266 nM); the others differ by up to a
factor of 75 (5-HT 9.2 nM against 687 nM). With the code's plateau (ii), 0 of 10; with Eq. 28 over
the recording (iii), 0 of 10; with the code's noise cut (iv), 2 of 10 (5-HT 632 nM, DA 1622 nM),
and the median distance from Table 1 falls from 0.37 to 0.12 log10 units. The median normalized
plateaus rise from 0 to about 1 over the wells of every group, while single cells reach -35 to 161,
and the pooled least-squares fits follow those cells.

**Result: not reproduced.** None of the four off-target EC50s is within 10% under any reading.
Readings (i) and (iv): AVP sensor to OXT 1339 and 558 nM against 28.7 uM; OXT sensor to AVP 32.5
and 20.2 nM against 247 nM; NE sensor to DA 562 and 2957 nM against 9.35 uM; DA sensor to NE 20.1
and 4.38 uM against 8.20 uM.

## table1-peak-dff-morse

> On MORSE probes the peak dF/F0 of each sensor to a saturating dose of its ligand is 0.37 (AVP),
> 0.24 (OXT), 0.36 (5-HT), 0.40 (CCK), 0.34 (NE), 0.27 (DA), 0.37 (ACh), 0.28 (ATP), 0.30 (MT) and
> 0.38 (HA), and to the partner ligand 0.10 (AVP sensor, OXT), 0.05 (OXT sensor, AVP), 0.15 (NE
> sensor, DA) and 0.21 (DA sensor, NE).

Identification recording, dF/F0 after the jump and background corrections; a cell's peak is the
maximum within the third minute of the well (morse_figs3_rev cell 160); the value is the median
over cells with a positive peak (cells 248, 263). Wells: (a) the ligand's well; (b) the code's
well (cell 248: the well after the ligand's for AVP, NE and CCK; for the OXT sensor's response to
AVP, the well after the AVP well).

In [ ]:
X = np.vstack([x[0] for x in pre])
EXP_PK = {("AVP", "AVP"): 0.37, ("AVP", "OXT"): 0.10, ("OXT", "OXT"): 0.24, ("OXT", "AVP"): 0.05, ("5HT", "5HT"): 0.36,
          ("CCK", "CCK"): 0.40, ("NE", "NE"): 0.34, ("NE", "DA"): 0.15, ("DA", "DA"): 0.27, ("DA", "NE"): 0.21,
          ("ACh", "ACh"): 0.37, ("ATP", "ATP"): 0.28, ("MT", "MT"): 0.30, ("HA", "HA"): 0.38}
W = {s: M.ID_WELLS.index(s) for s in M.ID_WELLS}
code_well = {("AVP", "AVP"): 1, ("NE", "NE"): 3, ("CCK", "CCK"): 9, ("OXT", "AVP"): 1}
rows = []
for g, e in EXP_PK.items():
    r = LAB == g[0]
    out = {"sensor": g[0], "ligand": g[1], "paper": e}
    for name, w in [("a", W[g[1]]), ("b", code_well.get(g, W[g[1]]))]:
        v = X[r][:, w * 93 + 62:w * 93 + 93].max(axis=1)
        pos = v[v > 0]
        out[f"({name}) well"] = M.ID_WELLS[w]
        out[f"({name}) median"] = round(float(np.median(pos)), 3)
        out[f"({name}) cells>0"] = f"{len(pos)}/{len(v)}"
        out[f"({name}) within 10%"] = abs(np.median(pos) - e) <= 0.1 * e + 1e-9
    rows.append(out)
PK = pd.DataFrame(rows)
print(PK.to_string(index=False))
for name in "ab":
    print(f"reading ({name}): {int(PK[f'({name}) within 10%'].sum())} of 14 within 10%")

**Result: partially reproduced.** 4 of 14 medians are within 10% of Table 1 in the ligand's well
(AVP 0.363, 5-HT 0.384, ACh 0.397, HA 0.400) and 5 of 14 in the code's wells (adding NE 0.338).
The others are higher than Table 1: OXT 0.292 against 0.24, CCK 0.492 against 0.40, DA 0.314
against 0.27, ATP 0.338 against 0.28, MT 0.564 against 0.30, and the crosstalk values 0.114, 0.044
(0.057 in the code's well), 0.207 and 0.236 against 0.10, 0.05, 0.15 and 0.21.

## fig1j-ec50-consistent-across-probes

> Dose-response relationships, after peak normalization and averaging over cells of the same
> sensor, are precise and consistent across probes.

Per-probe EC50s are fitted as in table1-ec50-on-target, reading (i), to each probe's cells alone,
for probes with more than one cell of the sensor (morse_figs3_rev cell 662). Readings (a) every
per-probe EC50 lies within a tenth to ten times the consensus EC50; (b) the SEM across probes is
less than half the mean.

In [ ]:
rows = []
for g in M.GROUP_ORDER:
    if g[0] != g[1]:
        continue
    cons = FITS[("i", g)][0]
    kds = []
    for p in range(10):
        r = (LAB == g[0]) & (grp == p)
        if r.sum() > 1:
            c, y = M.group_points(PLm[r], g, "code")
            kds.append(M.fit_group(c, y)[0])
    kds = np.array(kds)
    kn = []
    for p in range(10):
        r = (LAB == g[0]) & (grp == p)
        if r.sum() > 1:
            keep = M.noise_keep(M.normalized_dose_trace(DX[r], g))
            if keep.sum() > 1:
                c, y = M.group_points(PLm[r][keep], g, "code")
                kn.append(M.fit_group(c, y)[0])
    kn = np.array(kn)
    consn = FITS[("iv", g)][0]
    mean, sem = kds.mean(), kds.std(ddof=1) / np.sqrt(len(kds))
    rows.append({"sensor": g[0], "probes": len(kds), "consensus nM": round(cons * 1e9, 1),
                 "per-probe nM": np.round(kds * 1e9, 0).tolist(), "mean nM": round(mean * 1e9, 0), "SEM nM": round(sem * 1e9, 0),
                 "(a) all within 10x": bool(np.all((kds > cons / 10) & (kds < cons * 10))),
                 "(a) probes within": int(np.sum((kds > cons / 10) & (kds < cons * 10))),
                 "(b) SEM < mean/2": bool(sem < mean / 2),
                 "noise cut: probes": len(kn),
                 "noise cut (a)": bool(np.all((kn > consn / 10) & (kn < consn * 10))),
                 "noise cut (b)": bool(kn.std(ddof=1) / np.sqrt(len(kn)) < kn.mean() / 2) if len(kn) > 1 else False})
J = pd.DataFrame(rows)
print(J.to_string(index=False))
print("reading (a) holds for", int(J["(a) all within 10x"].sum()), "of 10 sensors; reading (b) for", int(J["(b) SEM < mean/2"].sum()), "of 10")
print("with the noise cut: (a)", int(J["noise cut (a)"].sum()), "of 10; (b)", int(J["noise cut (b)"].sum()), "of 10")

**Result: not reproduced.** Every per-probe EC50 lies within a tenth to ten times the consensus
for 2 of 10 sensors (ACh, HA; reading a), and the SEM is below half the mean for 6 of 10 (reading
b). With the code's noise cut, 7 and 8 of 10. Neither reading holds for all sensors with either
set of cells, and the consensus EC50s themselves do not reproduce Table 1.

## figs3e-tau-on-under-one-minute

> Ligands diffuse quickly through the hydrogel; almost all sensors show an onset time constant
> under 1 minute, and most signals plateau within 1 minute.

Per cell, the dose trace normalized as in reading (i), a 31-frame median filter, and an
exponential rise fitted to frames 0-78 of the well with the lowest concentration above the EC50
(morse_figs3_rev cells 688-692), on the normalized dF/F0 and on the Eq. 8 concentration. The
EC50 and F are the claims file's CONSTANTS (Table 1 and the code's F), since the EC50s fitted here
do not reproduce Table 1; the fitted EC50s of reading (i) are a variant. tau above 200 frames is
dropped. Seconds = frames x 60/31.

In [ ]:
rows = []
TAUS = {}
for g in M.GROUP_ORDER:
    if g[0] != g[1]:
        continue
    r = LAB == g[0]
    xb = M.normalized_dose_trace(DX[r], g)
    out = {"sensor": g[0], "cells": int(r.sum())}
    for name, kd, f in [("const", M.KD_TABLE1[g], M.F_CODE[g]), ("fitted", *FITS[("i", g)])]:
        td, tc, w = M.taus_for_group(xb, g, kd, f)
        if name == "const":
            TAUS[g] = (td, tc)
        out[f"{name} well"] = w
        out[f"{name} dF/F s"] = round(float(np.nanmedian(td)) * 60 / 31, 1) if len(td) else np.nan
        out[f"{name} conc s"] = round(float(np.nanmedian(tc)) * 60 / 31, 1) if len(tc) else np.nan
        out[f"{name} kept"] = f"{int(np.isfinite(td).sum())}/{int(np.isfinite(tc).sum())}"
    rows.append(out)
TE = pd.DataFrame(rows)
print(TE.to_string(index=False))
for name in ["const", "fitted"]:
    print(f"{name}: sensors with median tau-on under 60 s, (a) dF/F0 {int((TE[f'{name} dF/F s'] < 60).sum())} of 10, "
          f"(b) concentration {int((TE[f'{name} conc s'] < 60).sum())} of 10")

**Result: reproduced, resting on the labelling and on the Table 1 constants.** The median tau-on
is under 60 s for 10 of 10 sensors on the normalized dF/F0 (28.6 to 53.7 s) and for 8 of 10 on the
concentration (34.4 to 58.9 s; AVP 74.2 s and 5-HT 62.3 s are above). Both readings meet "at least
8 of 10". With the EC50s fitted here in place of Table 1's, 8 and 7 of 10.

## figs3f-depth-delay

> Onset times increase only modestly with cell depth in the gel, with a positive slope for all
> sensors except ATP.

The concentration tau-on of figs3e against each cell's depth: the z centre of mass of its
reordered mask in the dose recording (the code uses the dose recording's masks, morse_figs3_rev
cell 707), relative to the probe's shallowest cell and converted at 9.89 um per plane. The code
plots depth as 9.89 x (15 - z) (cell 718), so depth from the front face grows as the plane index
falls; this orientation is taken from the code, and the opposite one is shown. A least-squares line
per sensor, pooled over probes.

In [ ]:
t0 = time.time()
ZD = {p: M.z_centre(M.mask_z_profiles(M.file_path(p, "wells101"))) for p in M.REV_PROBES}
print(f"dose masks read in {time.time() - t0:.0f} s; masks per probe:", {p: len(v) for p, v in ZD.items()})
zrel = np.concatenate([ZD[p] - ZD[p].min() for p in M.REV_PROBES])
depth = -M.UM_PER_PLANE * zrel
print("depth range (um, 0 = deepest-indexed plane):", round(depth.min(), 1), "to", round(depth.max(), 1))
rows = []
for g in M.GROUP_ORDER:
    if g[0] != g[1]:
        continue
    r = np.where(LAB == g[0])[0]
    tc = TAUS[g][1] * 60 / 31
    ok = np.isfinite(tc)
    sl = np.polyfit(depth[r][ok], tc[ok], 1)[0]
    rho = np.corrcoef(depth[r][ok], tc[ok])[0, 1]
    rows.append({"sensor": g[0], "cells": int(ok.sum()), "slope s/um (code orientation)": round(sl, 3),
                 "slope s/um (opposite)": round(-sl, 3), "r": round(rho, 2)})
DF = pd.DataFrame(rows)
print(DF.to_string(index=False))
pos = DF["slope s/um (code orientation)"] > 0
print("positive slopes, code orientation:", int(pos.sum()), "of 10; negative:", DF.sensor[~pos].tolist())
print("positive slopes, opposite orientation:", int((~pos).sum()), "of 10")
print("largest |slope| (s/um):", DF["slope s/um (code orientation)"].abs().max())

**Result: not reproduced.** tau-on (concentration) rises with depth for 5 of 10 sensors (AVP,
5-HT, NE, MT, HA) and falls for 5 (OXT, CCK, DA, ACh, ATP) in the code's orientation of depth, and
the opposite orientation gives the same split. The slopes are modest (at most 0.48 s per um), and
the correlations weak (r = 0.56 for MT, at most 0.20 in size for the other nine).

## figs2f-cells-tracked

> About 70 to 130 active sensor cells per probe can be tracked across 2 to 8 recordings in
> different contexts.

A `matched<k>` file holds the masks matched between a reference recording and one other recording
of the probe (the k-th pair), and both files of a pair hold the same number of cells (shown in the
data check above for the revision probes). The matched count of a pair is that number. Files
deposited twice are counted once, and the refit copies (`-refit-matched`, the same masks) are left
out. Per probe, the mean over its pairs.

In [ ]:
Mt = S[S.kind.str.match(r"^mr-raw-matched\d*$") & (S.subject != "0")].copy()
Mt["probe"] = Mt.subject.str.replace(r"-(wells\d*|slice|mouse\d?)$", "", regex=True)
Mt = Mt[Mt.subject != "250918-SL1ZS5-wells001"]
Mt["pair"] = Mt.kind.str.extract(r"matched(\d*)$")[0].replace("", "0").astype(int)
chk = Mt.groupby(["probe", "pair"]).agg(n_cells=("cells", "nunique"), counts=("cells", lambda c: sorted(set(c))),
                                        files=("subject", lambda c: sorted(c)))
print("probe and pair suffixes whose files disagree on the count:")
print(chk[chk.n_cells > 1].to_string())
# The reference recording of a probe holds one matched<k> file per pair; take the pairs from it.
nref = Mt.groupby(["probe", "subject"]).size().rename("n").reset_index()
ref = nref.sort_values(["probe", "n", "subject"], ascending=[True, False, True]).groupby("probe").head(1)
pairs = Mt.merge(ref[["probe", "subject"]], on=["probe", "subject"]).groupby(["probe", "pair"]).agg(
    cells=("cells", "first"), contexts=("context", "first"))
ctx = Mt.groupby("probe").context.agg(lambda c: "+".join(sorted(set(c))))
pairs = pairs.reset_index().merge(ctx.rename("ctx").reset_index(), on="probe").set_index(["probe", "pair"])
pairs["contexts"] = pairs["ctx"]
print("\npairs (from each probe's reference recording):", len(pairs))
pp = pairs.reset_index().groupby("probe").agg(pairs=("pair", "count"), mean_matched=("cells", "mean"),
                                                 min_matched=("cells", "min"), max_matched=("cells", "max"),
                                                 contexts=("contexts", lambda c: ",".join(sorted(set(c)))))
pp["kind"] = np.where(pp.contexts.str.contains("mouse"), "in vivo", np.where(pp.contexts.str.contains("slice"), "slice", "dipping"))
inr = (pp.mean_matched >= 63) & (pp.mean_matched <= 143)
print(f"\nprobes: {len(pp)}; per-probe mean matched cells: median {pp.mean_matched.median():.0f}, "
      f"range {pp.mean_matched.min():.0f} to {pp.mean_matched.max():.0f}")
print(f"probes whose mean lies in 63-143: {int(inr.sum())} of {len(pp)} ({inr.mean():.2f})")
print(pp.assign(in_range=inr).groupby("kind").agg(probes=("pairs", "count"), pairs=("pairs", "sum"),
      median=("mean_matched", "median"), lo=("mean_matched", "min"), hi=("mean_matched", "max"), in_range=("in_range", "sum")).to_string())
print("\nrecordings per probe (pairs + 1): range", pp.pairs.min() + 1, "to", pp.pairs.max() + 1)
code_set = [p for p in pp.index if p.startswith("202309")]
cs = pp.loc[code_set]
print(f"\nthe code's dipping set (morse_figs2_2 cell 2, probes of 2023-09-15 to 2023-09-27): {len(cs)} probes, {cs.pairs.sum()} pairs; "
      f"per-probe means {np.round(cs.mean_matched.values, 1).tolist()}; in 63-143: {int(((cs.mean_matched >= 63) & (cs.mean_matched <= 143)).sum())} of {len(cs)}")

**Result: reproduced.** Over 72 probes and 110 matched pairs, 44 probes (0.61) have a mean
matched count within 63 to 143 (median 68, range 10 to 140), with 2 to 8 recordings per probe. By
context, 15 of 21 dipping probes, 24 of 40 in vivo and 5 of 11 slice probes are in range. In the
code's dipping set of 10 probes and 40 pairs (the n=40 of Figure S2F), 8 of 10 are in range (means
56.5 to 108.0).

## Summary

| Claim | Outcome |
|---|---|
| link-revision-well-order | reproduced |
| fig1c-cells-per-gel | reproduced (reading b; assumption on the gel thickness) |
| fig1g-sensor-yield | partially reproduced (assumed labelling) |
| fig1g-all-ten-sensors-respond | partially reproduced (assumed labelling) |
| fig1h-crosstalk-pairs | partially reproduced (assumed labelling) |
| table1-ec50-on-target | not reproduced (assumed labelling and normalization) |
| table1-ec50-off-target | not reproduced (assumed labelling and normalization) |
| table1-peak-dff-morse | partially reproduced (assumed labelling) |
| fig1j-ec50-consistent-across-probes | not reproduced (assumed labelling and normalization) |
| figs3e-tau-on-under-one-minute | reproduced (assumed labelling; Table 1 constants) |
| figs3f-depth-delay | not reproduced (assumed labelling; depth orientation from the code) |
| figs2f-cells-tracked | reproduced |
| fig1d, fig1f, methods-revision-probe-stability, figs2c, slices-probe-stability | blocked: data missing |
| figs6ef, fig4-ex-vivo-elevations, figs6j, fig5gh, fig6, fig6c, figs6a | blocked: data missing |
| fig4-ex-vivo-two-orders-lower | blocked: data missing (depends on fig4-ex-vivo-elevations) |
| the 16 slice claims, fig3df, fig3g, fig3h, fig4-ip-csf-concentrations, fig5ab, fig5cd | blocked: metadata insufficient |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The sensor labels are made by the Methods'
clustering without the code's hand corrections, so the labelled cells are not the paper's 506. The
dose-response normalization follows the code's baseline and peak wells, since the Methods do not
fix it, without the code's per-cell background fits over hand-set windows; the code's noise cut is
shown as a variant. The time base is the code's 31 frames per minute, since the files' 30 Hz is a
placeholder. tau-on uses the Table 1 EC50s to pick the well and convert to concentration.

What was found about the dataset. The files hold traces and masks only: no sensor labels, no
well, wash-on, injection or stimulation times, and no region or experiment labels, so the slice and
in vivo claims cannot be tested. The timing metadata are placeholders (30 Hz, every file starting
at 0 on 2023-01-01), and the indicator is given as OGB-1. The 3D masks are stored scrambled
(Fortran-order data in a C-order array). 67 files of an unnamed subject `0` carry no context; 16
of them repeat named files. The slice and wells001 files of 250918-SL1ZS5 are identical. In
20230922-PL28 the matched-file suffixes do not match their pairs (wells201 holds `matched` and
`matched1` as copies, wells401's `matched` file has the count of wells001's `matched3`), and the
251119-ZS16 pair has 143 against 142 cells.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")